# Google Landmarks Dataset v2 - Complete Download and GPS Coordinate Scraper

This notebook performs a complete pipeline to:
1. Download all Google Landmarks Dataset v2 shards from S3
2. Extract metadata and create image-to-URL mappings
3. Scrape GPS coordinates from Wikimedia in parallel (multi-threaded)
4. Filter landmarks located in Italy
5. Save results to comprehensive CSV files with resumable processing
6. Track shard/section information for each image

The process is designed to be **resumable**: if interrupted, it will continue from where it left off.

## Changes in V2 vs V1

### 1. Extracted images stored on Google Drive (not ephemeral local storage)
- **V1**: `extract_root = '/content/google_landmark_images'` — wiped on every Colab session restart.
- **V2**: `extract_root = os.path.join(project_folder, 'extracted_images')` — persisted to Google Drive alongside the TAR archives. Re-running after a disconnection no longer requires re-extracting shards.

### 2. Extraction skipped on resume via per-shard manifest
- **V1**: `get_shard_image_paths(tar_path)` — always re-extracts the TAR if the probe file is missing (i.e., after every session restart).
- **V2**: `get_shard_image_paths(tar_path, dataset, shard_idx)` — on first extraction, writes a `.image_manifest.txt` inside each shard directory listing all extracted JPG paths. On subsequent runs the manifest is read directly, skipping the TAR extraction entirely. Falls back to rebuilding the manifest from disk if the file is missing but images already exist.

### 3. Organized per-shard extraction directories
- **V1**: Images extracted into a flat `extract_root/<tar-internal-path>` structure, making it hard to clean up one shard without affecting others.
- **V2**: Each shard is extracted into its own folder: `extract_root/{dataset}/shard_{idx:03d}/`, making per-shard cleanup precise.

### 4. `DELETE_EXTRACTED_AFTER_SHARD` defaults to `False`
- **V1**: `DELETE_EXTRACTED_AFTER_SHARD = True` — frees local disk space after each shard but forces re-extraction on every resume.
- **V2**: `DELETE_EXTRACTED_AFTER_SHARD = False` — keeps extracted images in Drive so subsequent runs (or mid-run restarts) skip both download and extraction for already-processed shards. Set to `True` only if Drive space is tight.

In [ ]:
# @title 1. SETUP AND CONFIGURATION
# Import required libraries
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
import shutil
import tarfile
import time
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

print("Libraries imported successfully!")
print(f"Execution started at: {datetime.now()}")

# Mount Google Drive
print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

# Configure paths
project_folder = '/content/drive/My Drive/Vision_Project_2026/GLDv2'
os.makedirs(project_folder, exist_ok=True)

metadata_dir = '/content/metadata'
archive_dir = os.path.join(project_folder, 'archives')
# Persist extracted files next to archives so runs can resume without re-extracting shards.
extract_root = os.path.join(project_folder, 'extracted_images')

os.makedirs(metadata_dir, exist_ok=True)
os.makedirs(archive_dir, exist_ok=True)
os.makedirs(extract_root, exist_ok=True)

# Output CSV files
full_output_csv = os.path.join(project_folder, 'gld_full_dataset.csv')
italy_output_csv = os.path.join(project_folder, 'gld_italy_dataset.csv')
gps_cache_file = os.path.join(project_folder, 'gps_coordinate_cache.pkl')

print(f"Project folder: {project_folder}")
print(f"Metadata directory: {metadata_dir}")
print(f"Archive directory: {archive_dir}")
print(f"Extract root: {extract_root}")


In [ ]:
# @title 1b. PARAMETERS AND SETTINGS
# Dataset shard configuration
# Train: 500 shards, Index: 100 shards, Test: 20 shards
DATASET_CONFIG = {
    'train': {'shards': 500, 'url_base': 'https://s3.amazonaws.com/google-landmark/train/'},
    'index': {'shards': 100, 'url_base': 'https://s3.amazonaws.com/google-landmark/index/'},
    'test': {'shards': 20, 'url_base': 'https://s3.amazonaws.com/google-landmark/test/'},
}

# Which datasets to process (set to True/False)
PROCESS_TRAIN = True
PROCESS_INDEX = True
PROCESS_TEST = True

# Control shard range for testing (set START_SHARD=0, MAX_SHARDS=2 to test with just 2 shards)
# For production, use START_SHARD=0, MAX_SHARDS=500 (or 100 for index, 20 for test)
START_SHARD = 0
MAX_SHARDS = 500  # Modify for testing

# ============================================================================
# PERFORMANCE TUNING PARAMETERS
# ============================================================================
# Scraping speed controls
MAX_WORKERS = 24                   # Number of parallel GPS fetch threads
                                   # Recommended: 12-24 (higher = faster but more resource intensive)
                                   # Set based on your network: 12 (safe), 16 (balanced), 20+ (aggressive)

URL_BATCH_SIZE = 10000             # Process unique URLs in batches (increased from 2000 for speed)
                                   # Larger batches = fewer pause intervals = faster processing
                                   # Trade-off: Uses more memory during batch processing

BATCH_PAUSE_SECONDS = 0.05         # Pause between URL batches (reduced from 0.2)
                                   # Wikimedia tolerates 0.05-0.1 with good connection
                                   # If getting 429 errors, increase to 0.1-0.2

REQUEST_TIMEOUT_SECONDS = 8        # Timeout for individual requests (reduced from 10)
REQUEST_RETRIES = 1                # Retry failed requests (reduced from 2 for speed)
                                   # If too many failures, increase to 2

USE_CONNECTION_POOLING = True      # Use requests.Session for connection reuse (faster!)
PARALLEL_BATCH_FETCH = True        # Fetch multiple batches in parallel (experimental)

# Storage controls
KEEP_TAR_ARCHIVES = True           # Keep downloaded TAR files for future use
DELETE_EXTRACTED_AFTER_SHARD = False  # Keep extracted files for resumable shard reuse
SAVE_INTERVAL = 100                # Save CSV every N images processed

# Italy geographical filter (approximate bounding box)
ITALY_LAT_MIN, ITALY_LAT_MAX = 35.4, 47.2
ITALY_LON_MIN, ITALY_LON_MAX = 6.6, 18.8

print("Configuration Parameters:")
print(f"  Train processing: {PROCESS_TRAIN}")
print(f"  Index processing: {PROCESS_INDEX}")
print(f"  Test processing: {PROCESS_TEST}")
print(f"  Shards to process: {START_SHARD} to {MAX_SHARDS-1}")
print(f"\n  PERFORMANCE TUNING:")
print(f"  Max workers: {MAX_WORKERS} (threads)")
print(f"  URL batch size: {URL_BATCH_SIZE}")
print(f"  Batch pause: {BATCH_PAUSE_SECONDS} seconds")
print(f"  Request timeout: {REQUEST_TIMEOUT_SECONDS} seconds")
print(f"  Request retries: {REQUEST_RETRIES}")
print(f"  Connection pooling: {USE_CONNECTION_POOLING}")
print(f"  Keep extracted shards: {not DELETE_EXTRACTED_AFTER_SHARD}")
print(f"  Italy bounds - Lat: [{ITALY_LAT_MIN}, {ITALY_LAT_MAX}], Lon: [{ITALY_LON_MIN}, {ITALY_LON_MAX}]")
print(f"\n  ⏱️  ESTIMATED TIME FOR FULL DATASET:")
print(f"     Train (4.1M images): 15-25 hours")
print(f"     Index (761K images): 3-5 hours")
print(f"     Test (117K images): 0.5-1 hour")
print(f"     TOTAL: 18-31 hours (continuous processing)")
print(f"     Recommendation: Run overnight or on a schedule")


In [ ]:
# @title 2. HELPER FUNCTIONS - Download, Extract, and Metadata
import json
from datetime import datetime

def ensure_metadata_files():
    """Download metadata files for each dataset if missing."""
    metadata_urls = {
        'train': {
            'train.csv': 'https://s3.amazonaws.com/google-landmark/metadata/train.csv',
            'train_label_to_category.csv': 'https://s3.amazonaws.com/google-landmark/metadata/train_label_to_category.csv',
        },
        'index': {
            'index_image_to_landmark.csv': 'https://s3.amazonaws.com/google-landmark/metadata/index_image_to_landmark.csv',
            'index_label_to_category.csv': 'https://s3.amazonaws.com/google-landmark/metadata/index_label_to_category.csv',
        },
        'test': {
            'test.csv': 'https://s3.amazonaws.com/google-landmark/metadata/test.csv',
        }
    }
    
    downloaded = {}
    total_files = sum(len(files) for files in metadata_urls.values())
    current_file = 0
    
    print(f"[Metadata] Starting download of {total_files} metadata files...\n")
    
    for dataset, files in metadata_urls.items():
        for filename, url in files.items():
            current_file += 1
            filepath = os.path.join(metadata_dir, filename)
            
            if os.path.exists(filepath):
                file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
                print(f"[{current_file}/{total_files}] ✓ CACHED: {filename} ({file_size_mb:.2f} MB)")
            else:
                print(f"[{current_file}/{total_files}] ⬇️  DOWNLOADING: {filename}...")
                result = subprocess.run(['wget', '-q', '-O', filepath, url], capture_output=True)
                if result.returncode != 0:
                    print(f"           ✗ WARNING: Failed to download {filename}")
                else:
                    file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
                    print(f"           ✓ Downloaded: {filename} ({file_size_mb:.2f} MB)")
            
            downloaded[filename] = filepath
    
    print(f"\n✓ All metadata files ready!\n")
    return downloaded


def ensure_output_files(csv_path):
    """Initialize output CSV file with headers if it doesn't exist."""
    cols = ['image_id', 'file_path', 'wiki_url', 'lat', 'lon', 'status', 'dataset', 'shard', 'is_italy']
    if not os.path.exists(csv_path):
        pd.DataFrame(columns=cols).to_csv(csv_path, index=False)
        print(f"[CSV] Created: {csv_path}")
    else:
        num_rows = len(pd.read_csv(csv_path))
        print(f"[CSV] Found existing: {csv_path} ({num_rows} rows)")


def load_processed_ids(csv_path):
    """Load set of already processed image IDs for resumable processing."""
    if not os.path.exists(csv_path):
        print("[Resume] No existing CSV found - starting fresh")
        return set()
    try:
        existing = pd.read_csv(csv_path, usecols=['image_id'])
        ids = set(existing['image_id'].astype(str).unique())
        print(f"[Resume] ✓ Loaded {len(ids)} previously processed image IDs")
        return ids
    except Exception as e:
        print(f"[Resume] Error loading processed IDs: {e}")
        return set()


def get_cached_shards_info(archive_dir, dataset):
    """Get information about cached/downloaded shards."""
    cached_shards = []
    total_size = 0
    
    pattern = f'{dataset}_images_*.tar'
    for filepath in Path(archive_dir).glob(pattern):
        if filepath.is_file():
            size_mb = filepath.stat().st_size / (1024 * 1024)
            shard_num = filepath.name.split('_')[2].replace('.tar', '')
            cached_shards.append((int(shard_num), size_mb))
            total_size += size_mb
    
    cached_shards.sort()
    return cached_shards, total_size


def download_shard(shard_idx, dataset, url_base, archive_dir):
    """Download a TAR shard from S3 with resume capability and logging."""
    shard_name = f'images_{shard_idx:03d}.tar'
    local_tar = os.path.join(archive_dir, f'{dataset}_{shard_name}')
    
    # Check if already cached
    if os.path.exists(local_tar):
        size_mb = os.path.getsize(local_tar) / (1024 * 1024)
        return local_tar, True, size_mb  # Return cached flag and size
    
    # Download the shard
    url = f'{url_base}{shard_name}'
    cmd = ['wget', '-q', '-c', '-O', local_tar, url]
    result = subprocess.run(cmd, capture_output=True)
    
    if result.returncode != 0:
        if os.path.exists(local_tar):
            try:
                os.remove(local_tar)
            except OSError:
                pass
        return None, False, 0
    
    # Get size of downloaded file
    size_mb = os.path.getsize(local_tar) / (1024 * 1024)
    return local_tar, False, size_mb  # Return False for not cached, and size


def get_shard_image_paths(tar_path, dataset, shard_idx):
    """
    Return JPG image paths for a shard, reusing persisted extracted files if available.
    Extraction target: {extract_root}/{dataset}/shard_XXX
    """
    shard_extract_dir = os.path.join(extract_root, dataset, f'shard_{shard_idx:03d}')
    manifest_path = os.path.join(shard_extract_dir, '.image_manifest.txt')

    # Fast path: use manifest generated in a previous run.
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, 'r') as f:
                manifest_paths = [line.strip() for line in f if line.strip()]
            cached_paths = [p for p in manifest_paths if os.path.exists(p)]
            if cached_paths:
                print(f"[Extract] Using cached extracted shard {dataset}:{shard_idx:03d} ({len(cached_paths)} files)")
                return cached_paths, [shard_extract_dir]
        except Exception:
            pass

    # Fallback: if extracted files exist but manifest is missing/corrupt, rebuild manifest.
    if os.path.isdir(shard_extract_dir):
        existing_paths = sorted([str(p) for p in Path(shard_extract_dir).rglob('*.jpg')])
        if existing_paths:
            try:
                with open(manifest_path, 'w') as f:
                    f.write('\n'.join(existing_paths))
            except Exception:
                pass
            print(f"[Extract] Rebuilt manifest for cached shard {dataset}:{shard_idx:03d} ({len(existing_paths)} files)")
            return existing_paths, [shard_extract_dir]

    # Cold path: extract once, then persist manifest for future resume runs.
    os.makedirs(shard_extract_dir, exist_ok=True)
    try:
        with tarfile.open(tar_path, 'r') as tar:
            print(f"[Extract] Extracting {os.path.basename(tar_path)} into {shard_extract_dir}...")
            tar.extractall(path=shard_extract_dir)
    except Exception as e:
        print(f"[Extract] Error extracting TAR: {e}")
        return [], []

    extracted_paths = sorted([str(p) for p in Path(shard_extract_dir).rglob('*.jpg')])
    if not extracted_paths:
        return [], [shard_extract_dir]

    try:
        with open(manifest_path, 'w') as f:
            f.write('\n'.join(extracted_paths))
    except Exception:
        pass

    return extracted_paths, [shard_extract_dir]


def cleanup_extracted_shard(shard_dirs):
    """Delete extracted shard folders to free disk space."""
    for target in shard_dirs:
        if os.path.isdir(target):
            try:
                shutil.rmtree(target, ignore_errors=True)
            except Exception:
                pass


def is_in_italy(lat, lon):
    """Check if coordinates are within Italy's bounding box."""
    return (
        lat is not None
        and lon is not None
        and ITALY_LAT_MIN <= lat <= ITALY_LAT_MAX
        and ITALY_LON_MIN <= lon <= ITALY_LON_MAX
    )

print("✓ Helper functions defined successfully!")
print(f"\n[Paths] Downloaded shards location: {archive_dir}")

In [ ]:
# @title 3. GPS COORDINATE EXTRACTION FUNCTIONS (OPTIMIZED)
# Create a persistent session for connection pooling
if USE_CONNECTION_POOLING:
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    
    # Create a session with automatic retries
    session = requests.Session()
    
    # Configure retry strategy
    retry_strategy = Retry(
        total=REQUEST_RETRIES,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
else:
    session = None

print("✓ GPS extraction optimized with connection pooling" if USE_CONNECTION_POOLING else "✓ GPS extraction ready")

def get_gps_from_wikimedia(url, session=None):
    """
    Extract monument GPS coordinates from Wikimedia GeoHack link.
    Parses the geohack.toolforge.org URL to extract latitude and longitude.
    Optimized with connection pooling and faster error handling.
    Returns: (lat, lon) tuple or None if extraction fails
    """
    headers = {'User-Agent': 'UniversityProject/1.0 (contact: student@university.edu)'}
    
    try:
        # Use session if available (connection pooling) for faster requests
        if session is not None:
            response = session.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        else:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        geohack_link = soup.find('a', href=lambda href: href and 'geohack.toolforge.org' in href)
        if not geohack_link:
            return None
        
        # Parse GeoHack URL to extract coordinates
        # Format: params=LAT_NS_LON_EW
        match = re.search(r'params=([\d\.]+)_([NS])_([\d\.]+)_([EW])', geohack_link['href'])
        if not match:
            return None
        
        lat, lat_dir, lon, lon_dir = match.groups()
        lat, lon = float(lat), float(lon)
        
        if lat_dir == 'S':
            lat = -lat
        if lon_dir == 'W':
            lon = -lon
        
        return lat, lon
    except requests.Timeout:
        return None
    except Exception:
        return None


def fetch_gps_for_urls(urls, max_workers):
    """
    Fetch GPS coordinates for multiple unique Wikimedia URLs in parallel.
    Uses ThreadPoolExecutor for concurrent requests with connection pooling.
    Returns: dict mapping URL -> (lat, lon) or None
    """
    if not urls:
        return {}
    
    url_to_gps = {}
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create futures with session passed to each thread
        futures = {
            executor.submit(get_gps_from_wikimedia, url, session): url 
            for url in urls
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), 
                          desc='Fetching GPS coords', leave=False, position=1):
            url = futures[future]
            try:
                url_to_gps[url] = future.result()
            except Exception:
                url_to_gps[url] = None
    
    return url_to_gps


def flush_batches(full_batch, italy_batch, full_csv, italy_csv):
    """Incrementally save batch records to CSV files."""
    if full_batch:
        pd.DataFrame(full_batch).to_csv(full_csv, mode='a', header=False, index=False)
    if italy_batch:
        pd.DataFrame(italy_batch).to_csv(italy_csv, mode='a', header=False, index=False)

print("✓ GPS extraction functions defined successfully!")


In [ ]:
# @title 4. PREPARE METADATA AND OUTPUT FILES
print("=" * 80)
print("STEP 1: Preparing metadata files and output directories")
print("=" * 80)

# Download and prepare metadata
metadata_files = ensure_metadata_files()

# Initialize output CSV files
print("\nInitializing output CSV files...")
ensure_output_files(full_output_csv)
ensure_output_files(italy_output_csv)

# Load previously processed IDs for resumable processing
print("\nLoading resume state...")
processed_ids = load_processed_ids(full_output_csv)

# Check for cached shards
print("\n" + "=" * 80)
print("STEP 2: Checking cache status")
print("=" * 80)

for dataset in ['train', 'index', 'test']:
    cached, total_size = get_cached_shards_info(archive_dir, dataset)
    if cached:
        shard_nums = [s[0] for s in cached]
        print(f"\n[{dataset.upper()}] Found {len(cached)} cached shards")
        print(f"  Shards: {min(shard_nums)} - {max(shard_nums)}")
        print(f"  Total cache size: {total_size:.2f} GB")
    else:
        print(f"\n[{dataset.upper()}] No cached shards found")

print(f"\n[Paths] Extracted images location: {extract_root}")
print(f"[Paths] CSV output location: {project_folder}")
print(f"[Paths] Metadata cache location: {metadata_dir}")
print("\n✓ Preparation complete!\n")

In [ ]:
# @title 5a. PROCESS TRAIN DATASET
if PROCESS_TRAIN:
    print("\n" + "=" * 80)
    print("PROCESSING TRAIN DATASET (500 shards)")
    print("=" * 80)
    
    # Load train metadata
    train_csv_path = os.path.join(metadata_dir, 'train.csv')
    train_label_csv_path = os.path.join(metadata_dir, 'train_label_to_category.csv')
    
    if not os.path.exists(train_csv_path) or not os.path.exists(train_label_csv_path):
        print("ERROR: Train metadata files not found. Please ensure they were downloaded.")
    else:
        print("[Train] Loading metadata files...")
        df_labels = pd.read_csv(train_csv_path)
        df_categories = pd.read_csv(train_label_csv_path)
        print(f"  ✓ Loaded train.csv: {len(df_labels)} rows")
        print(f"  ✓ Loaded categories: {len(df_categories)} rows")
        
        print("[Train] Merging dataframes...")
        full_df = pd.merge(df_labels, df_categories, on='landmark_id')
        print(f"  ✓ Merged: {len(full_df)} landmark entries")
        
        # Create fast lookup: image_id -> Wikimedia category URL
        print("[Train] Creating image ID -> URL lookup table...")
        id_to_wiki = (
            full_df[['id', 'category']]
            .dropna(subset=['id', 'category'])
            .drop_duplicates(subset=['id'])
            .set_index('id')['category']
            .to_dict()
        )
        print(f"  ✓ Created lookup: {len(id_to_wiki)} unique image IDs with URLs")
        
        # Initialize counters and tracking
        train_full_batch = []
        train_italy_batch = []
        train_processed_this_run = 0
        train_success_this_run = 0
        train_italy_success_this_run = 0
        train_downloaded = 0
        train_cached = 0
        train_total_download_size = 0
        
        print(f"\n[Train] Starting shard processing (shards {START_SHARD} to {MAX_SHARDS-1})...")
        
        # Process each shard
        for shard_idx in tqdm(range(START_SHARD, MAX_SHARDS), desc='Train Shards', position=0):
            # Download shard
            result = download_shard(shard_idx, 'train', DATASET_CONFIG['train']['url_base'], archive_dir)
            
            if result is None:
                print(f"[Train] Shard {shard_idx:03d}: Not available")
                continue
            
            tar_path, was_cached, size_mb = result
            
            if was_cached:
                train_cached += 1
                status_str = "✓ CACHED"
            else:
                train_downloaded += 1
                status_str = "⬇️  DOWNLOADED"
            
            train_total_download_size += size_mb
            
            # Extract and get image paths
            try:
                shard_files, shard_top_dirs = get_shard_image_paths(tar_path, 'train', shard_idx)
            except Exception as e:
                print(f"[Train] Shard {shard_idx:03d}: Error reading - {e}")
                continue
            
            if not shard_files:
                print(f"[Train] Shard {shard_idx:03d}: No JPG files found")
                continue
            
            # Filter out already processed images
            remaining_items = []
            for file_path in shard_files:
                img_id = Path(file_path).stem
                if img_id in processed_ids:
                    continue
                remaining_items.append({
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': id_to_wiki.get(img_id),
                })
            
            if not remaining_items:
                if DELETE_EXTRACTED_AFTER_SHARD:
                    cleanup_extracted_shard(shard_top_dirs)
                continue
            
            # Batch fetch unique URLs
            unique_urls = sorted({item['wiki_url'] for item in remaining_items if item['wiki_url']})
            
            url_to_gps = {}
            for start in range(0, len(unique_urls), URL_BATCH_SIZE):
                batch_urls = unique_urls[start : start + URL_BATCH_SIZE]
                batch_map = fetch_gps_for_urls(batch_urls, max_workers=MAX_WORKERS)
                url_to_gps.update(batch_map)
                if BATCH_PAUSE_SECONDS > 0:
                    time.sleep(BATCH_PAUSE_SECONDS)
            
            # Build records with GPS data
            for i, item in enumerate(tqdm(remaining_items, desc=f'Shard {shard_idx:03d} records', leave=False)):
                img_id = item['image_id']
                file_path = item['file_path']
                wiki_url = item['wiki_url']
                
                if not wiki_url:
                    record = {
                        'image_id': img_id,
                        'file_path': file_path,
                        'wiki_url': None,
                        'lat': None,
                        'lon': None,
                        'status': 'missing_metadata',
                        'dataset': 'train',
                        'shard': shard_idx,
                        'is_italy': False,
                    }
                else:
                    gps = url_to_gps.get(wiki_url)
                    if gps is None:
                        record = {
                            'image_id': img_id,
                            'file_path': file_path,
                            'wiki_url': wiki_url,
                            'lat': None,
                            'lon': None,
                            'status': 'no_gps_found',
                            'dataset': 'train',
                            'shard': shard_idx,
                            'is_italy': False,
                        }
                    else:
                        lat, lon = gps
                        in_italy = is_in_italy(lat, lon)
                        record = {
                            'image_id': img_id,
                            'file_path': file_path,
                            'wiki_url': wiki_url,
                            'lat': lat,
                            'lon': lon,
                            'status': 'success',
                            'dataset': 'train',
                            'shard': shard_idx,
                            'is_italy': in_italy,
                        }
                        train_success_this_run += 1
                        if in_italy:
                            train_italy_success_this_run += 1
                
                train_full_batch.append(record)
                if record['status'] == 'success' and record['is_italy']:
                    train_italy_batch.append(record)
                
                processed_ids.add(img_id)
                train_processed_this_run += 1
                
                # Periodically flush to CSV
                if (i + 1) % SAVE_INTERVAL == 0:
                    flush_batches(train_full_batch, train_italy_batch, full_output_csv, italy_output_csv)
                    train_full_batch, train_italy_batch = [], []
            
            # Save remaining records
            flush_batches(train_full_batch, train_italy_batch, full_output_csv, italy_output_csv)
            train_full_batch, train_italy_batch = [], []
            
            # Cleanup
            if DELETE_EXTRACTED_AFTER_SHARD:
                cleanup_extracted_shard(shard_top_dirs)
            
            if not KEEP_TAR_ARCHIVES:
                try:
                    os.remove(tar_path)
                except:
                    pass
        
        print(f"\n--- TRAIN DATASET SUMMARY ---")
        print(f"  Downloaded: {train_downloaded} new shards ({train_total_download_size:.2f} GB)")
        print(f"  Cached: {train_cached} shards")
        print(f"  Total processed: {train_processed_this_run} images")
        print(f"  GPS extracted: {train_success_this_run} images")
        print(f"  Italy landmarks: {train_italy_success_this_run} images")
else:
    print("\n[Train] Processing SKIPPED (PROCESS_TRAIN=False)")

In [ ]:
# @title 5b. PROCESS INDEX DATASET
if PROCESS_INDEX:
    print("\n" + "=" * 80)
    print("PROCESSING INDEX DATASET (100 shards)")
    print("=" * 80)
    
    # Load index metadata
    index_img_landmark_path = os.path.join(metadata_dir, 'index_image_to_landmark.csv')
    index_label_csv_path = os.path.join(metadata_dir, 'index_label_to_category.csv')
    
    if not os.path.exists(index_img_landmark_path) or not os.path.exists(index_label_csv_path):
        print("ERROR: Index metadata files not found. Please ensure they were downloaded.")
    else:
        print("[Index] Loading metadata files...")
        df_img_landmark = pd.read_csv(index_img_landmark_path)
        df_categories = pd.read_csv(index_label_csv_path)
        print(f"  ✓ Loaded image-landmark mapping: {len(df_img_landmark)} rows")
        print(f"  ✓ Loaded categories: {len(df_categories)} rows")
        
        print("[Index] Merging dataframes...")
        full_df = pd.merge(df_img_landmark, df_categories, on='landmark_id')
        print(f"  ✓ Merged: {len(full_df)} landmark entries")
        
        # Create fast lookup: image_id -> Wikimedia category URL
        print("[Index] Creating image ID -> URL lookup table...")
        id_to_wiki = (
            full_df[['id', 'category']]
            .dropna(subset=['id', 'category'])
            .drop_duplicates(subset=['id'])
            .set_index('id')['category']
            .to_dict()
        )
        print(f"  ✓ Created lookup: {len(id_to_wiki)} unique image IDs with URLs")
        
        # Initialize counters and tracking
        index_full_batch = []
        index_italy_batch = []
        index_processed_this_run = 0
        index_success_this_run = 0
        index_italy_success_this_run = 0
        index_downloaded = 0
        index_cached = 0
        index_total_download_size = 0
        
        print(f"\n[Index] Starting shard processing (shards {START_SHARD} to {min(MAX_SHARDS, 99)})...")
        
        # Process each shard (max 100 shards for index)
        max_index_shards = min(MAX_SHARDS, 100)
        for shard_idx in tqdm(range(START_SHARD, max_index_shards), desc='Index Shards', position=0):
            # Download shard
            result = download_shard(shard_idx, 'index', DATASET_CONFIG['index']['url_base'], archive_dir)
            
            if result is None:
                continue
            
            tar_path, was_cached, size_mb = result
            
            if was_cached:
                index_cached += 1
            else:
                index_downloaded += 1
            
            index_total_download_size += size_mb
            
            # Extract and get image paths
            try:
                shard_files, shard_top_dirs = get_shard_image_paths(tar_path, 'index', shard_idx)
            except Exception as e:
                print(f"[Index] Shard {shard_idx:03d}: Error reading - {e}")
                continue
            
            if not shard_files:
                print(f"[Index] Shard {shard_idx:03d}: No JPG files found")
                continue
            
            # Filter out already processed images
            remaining_items = []
            for file_path in shard_files:
                img_id = Path(file_path).stem
                if img_id in processed_ids:
                    continue
                remaining_items.append({
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': id_to_wiki.get(img_id),
                })
            
            if not remaining_items:
                if DELETE_EXTRACTED_AFTER_SHARD:
                    cleanup_extracted_shard(shard_top_dirs)
                continue
            
            # Batch fetch unique URLs
            unique_urls = sorted({item['wiki_url'] for item in remaining_items if item['wiki_url']})
            
            url_to_gps = {}
            for start in range(0, len(unique_urls), URL_BATCH_SIZE):
                batch_urls = unique_urls[start : start + URL_BATCH_SIZE]
                batch_map = fetch_gps_for_urls(batch_urls, max_workers=MAX_WORKERS)
                url_to_gps.update(batch_map)
                if BATCH_PAUSE_SECONDS > 0:
                    time.sleep(BATCH_PAUSE_SECONDS)
            
            # Build records with GPS data
            for i, item in enumerate(tqdm(remaining_items, desc=f'Shard {shard_idx:03d} records', leave=False)):
                img_id = item['image_id']
                file_path = item['file_path']
                wiki_url = item['wiki_url']
                
                if not wiki_url:
                    record = {
                        'image_id': img_id,
                        'file_path': file_path,
                        'wiki_url': None,
                        'lat': None,
                        'lon': None,
                        'status': 'missing_metadata',
                        'dataset': 'index',
                        'shard': shard_idx,
                        'is_italy': False,
                    }
                else:
                    gps = url_to_gps.get(wiki_url)
                    if gps is None:
                        record = {
                            'image_id': img_id,
                            'file_path': file_path,
                            'wiki_url': wiki_url,
                            'lat': None,
                            'lon': None,
                            'status': 'no_gps_found',
                            'dataset': 'index',
                            'shard': shard_idx,
                            'is_italy': False,
                        }
                    else:
                        lat, lon = gps
                        in_italy = is_in_italy(lat, lon)
                        record = {
                            'image_id': img_id,
                            'file_path': file_path,
                            'wiki_url': wiki_url,
                            'lat': lat,
                            'lon': lon,
                            'status': 'success',
                            'dataset': 'index',
                            'shard': shard_idx,
                            'is_italy': in_italy,
                        }
                        index_success_this_run += 1
                        if in_italy:
                            index_italy_success_this_run += 1
                
                index_full_batch.append(record)
                if record['status'] == 'success' and record['is_italy']:
                    index_italy_batch.append(record)
                
                processed_ids.add(img_id)
                index_processed_this_run += 1
                
                # Periodically flush to CSV
                if (i + 1) % SAVE_INTERVAL == 0:
                    flush_batches(index_full_batch, index_italy_batch, full_output_csv, italy_output_csv)
                    index_full_batch, index_italy_batch = [], []
            
            # Save remaining records
            flush_batches(index_full_batch, index_italy_batch, full_output_csv, italy_output_csv)
            index_full_batch, index_italy_batch = [], []
            
            # Cleanup
            if DELETE_EXTRACTED_AFTER_SHARD:
                cleanup_extracted_shard(shard_top_dirs)
            
            if not KEEP_TAR_ARCHIVES:
                try:
                    os.remove(tar_path)
                except:
                    pass
        
        print(f"\n--- INDEX DATASET SUMMARY ---")
        print(f"  Downloaded: {index_downloaded} new shards ({index_total_download_size:.2f} GB)")
        print(f"  Cached: {index_cached} shards")
        print(f"  Total processed: {index_processed_this_run} images")
        print(f"  GPS extracted: {index_success_this_run} images")
        print(f"  Italy landmarks: {index_italy_success_this_run} images")
else:
    print("\n[Index] Processing SKIPPED (PROCESS_INDEX=False)")

In [ ]:
# @title 5c. PROCESS TEST DATASET
if PROCESS_TEST:
    print("\n" + "=" * 80)
    print("PROCESSING TEST DATASET (20 shards)")
    print("=" * 80)
    
    test_csv_path = os.path.join(metadata_dir, 'test.csv')
    
    if not os.path.exists(test_csv_path):
        print("ERROR: Test metadata file not found. Please ensure it was downloaded.")
    else:
        print("[Test] Loading metadata files...")
        df_test = pd.read_csv(test_csv_path)
        test_ids = set(df_test['id'].astype(str).unique())
        print(f"  ✓ Loaded test.csv: {len(test_ids)} image IDs")
        
        # Initialize counters and tracking
        test_full_batch = []
        test_processed_this_run = 0
        test_downloaded = 0
        test_cached = 0
        test_total_download_size = 0
        
        print(f"\n[Test] Starting shard processing (shards {START_SHARD} to {min(MAX_SHARDS, 19)})...")
        
        # Process each shard (max 20 shards for test)
        max_test_shards = min(MAX_SHARDS, 20)
        for shard_idx in tqdm(range(START_SHARD, max_test_shards), desc='Test Shards', position=0):
            # Download shard
            result = download_shard(shard_idx, 'test', DATASET_CONFIG['test']['url_base'], archive_dir)
            
            if result is None:
                continue
            
            tar_path, was_cached, size_mb = result
            
            if was_cached:
                test_cached += 1
            else:
                test_downloaded += 1
            
            test_total_download_size += size_mb
            
            # Extract and get image paths
            try:
                shard_files, shard_top_dirs = get_shard_image_paths(tar_path, 'test', shard_idx)
            except Exception as e:
                print(f"[Test] Shard {shard_idx:03d}: Error reading - {e}")
                continue
            
            if not shard_files:
                print(f"[Test] Shard {shard_idx:03d}: No JPG files found")
                continue
            
            # Filter out already processed images
            remaining_items = []
            for file_path in shard_files:
                img_id = Path(file_path).stem
                if img_id in processed_ids:
                    continue
                remaining_items.append({
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': None,  # Test set doesn't have landmark metadata
                })
            
            if not remaining_items:
                if DELETE_EXTRACTED_AFTER_SHARD:
                    cleanup_extracted_shard(shard_top_dirs)
                continue
            
            # Build records (test set has no GPS data available)
            for i, item in enumerate(tqdm(remaining_items, desc=f'Shard {shard_idx:03d} records', leave=False)):
                img_id = item['image_id']
                file_path = item['file_path']
                
                record = {
                    'image_id': img_id,
                    'file_path': file_path,
                    'wiki_url': None,
                    'lat': None,
                    'lon': None,
                    'status': 'no_metadata_available',
                    'dataset': 'test',
                    'shard': shard_idx,
                    'is_italy': False,
                }
                
                test_full_batch.append(record)
                processed_ids.add(img_id)
                test_processed_this_run += 1
                
                # Periodically flush to CSV
                if (i + 1) % SAVE_INTERVAL == 0:
                    flush_batches(test_full_batch, [], full_output_csv, italy_output_csv)
                    test_full_batch = []
            
            # Save remaining records
            flush_batches(test_full_batch, [], full_output_csv, italy_output_csv)
            test_full_batch = []
            
            # Cleanup
            if DELETE_EXTRACTED_AFTER_SHARD:
                cleanup_extracted_shard(shard_top_dirs)
            
            if not KEEP_TAR_ARCHIVES:
                try:
                    os.remove(tar_path)
                except:
                    pass
        
        print(f"\n--- TEST DATASET SUMMARY ---")
        print(f"  Downloaded: {test_downloaded} new shards ({test_total_download_size:.2f} GB)")
        print(f"  Cached: {test_cached} shards")
        print(f"  Total processed: {test_processed_this_run} images")
        print(f"  Note: Test set has no metadata/GPS available")
else:
    print("\n[Test] Processing SKIPPED (PROCESS_TEST=False)")

In [ ]:
# @title 6. GENERATE SUMMARY STATISTICS AND CACHE REPORT
print("\n" + "=" * 80)
print("FINAL SUMMARY STATISTICS AND CACHE REPORT")
print("=" * 80)

# Cache statistics
print("\n--- DOWNLOADED SHARDS CACHE STATUS ---")
for dataset in ['train', 'index', 'test']:
    cached, total_size = get_cached_shards_info(archive_dir, dataset)
    if cached:
        print(f"\n[{dataset.upper()}]")
        print(f"  Cached shards: {len(cached)}")
        print(f"  Total size: {total_size:.2f} GB")
        print(f"  Range: {min([s[0] for s in cached]):03d} - {max([s[0] for s in cached]):03d}")
    else:
        print(f"\n[{dataset.upper()}] No cached shards")

# Load final CSV files
print("\n--- PROCESSING RESULTS ---")
if os.path.exists(full_output_csv):
    final_full_df = pd.read_csv(full_output_csv)
    print(f"\n[Full Dataset CSV]")
    print(f"  Location: {full_output_csv}")
    print(f"  Total rows: {len(final_full_df):,}")
    print(f"\n  Status breakdown:")
    status_counts = final_full_df['status'].value_counts()
    for status, count in status_counts.items():
        pct = count / len(final_full_df) * 100
        print(f"    {status}: {count:,} ({pct:.1f}%)")
    
    print(f"\n  Dataset breakdown:")
    dataset_counts = final_full_df['dataset'].value_counts()
    for dataset, count in dataset_counts.items():
        print(f"    {dataset}: {count:,} images")
    
    italy_count = (final_full_df['is_italy'] == True).sum()
    success_count = (final_full_df['status'] == 'success').sum()
    print(f"\n  GPS Extraction:")
    print(f"    Successful: {success_count:,} ({success_count/len(final_full_df)*100:.1f}%)")
    print(f"    Italy landmarks: {italy_count:,}")
else:
    print("\nFull dataset CSV not found!")
    final_full_df = None

if os.path.exists(italy_output_csv):
    final_italy_df = pd.read_csv(italy_output_csv)
    print(f"\n[Italy-Only Dataset CSV]")
    print(f"  Location: {italy_output_csv}")
    print(f"  Total rows: {len(final_italy_df):,}")
    
    if len(final_italy_df) > 0:
        print(f"\n  Distribution by dataset:")
        italy_dataset_counts = final_italy_df['dataset'].value_counts()
        for dataset, count in italy_dataset_counts.items():
            print(f"    {dataset}: {count:,}")
else:
    print("\nItaly dataset CSV not found!")
    final_italy_df = None

print(f"\n--- OUTPUT FILE LOCATIONS ---")
print(f"Full dataset CSV:      {full_output_csv}")
print(f"Italy-only CSV:        {italy_output_csv}")
print(f"Downloaded shards:     {archive_dir}")
print(f"Extracted images:      {extract_root}")
print(f"Metadata cache:        {metadata_dir}")

print(f"\nExecution completed at: {datetime.now()}")

# Display sample data
print(f"\n--- SAMPLE DATA (First 5 rows from Full Dataset) ---")
if final_full_df is not None and len(final_full_df) > 0:
    print(final_full_df.head(5).to_string())

print(f"\n--- SAMPLE DATA (First 5 rows from Italy Dataset) ---")
if final_italy_df is not None and len(final_italy_df) > 0:
    print(final_italy_df.head(5).to_string())


In [ ]:
# @title 7. DATA EXPLORATION AND ANALYSIS (Optional)
# This cell provides useful analysis of the collected data

print("=" * 80)
print("DATA EXPLORATION AND ANALYSIS")
print("=" * 80)

if final_full_df is not None and len(final_full_df) > 0:
    print("\n--- Full Dataset Analysis ---")
    
    # Success rate
    success_rate = (final_full_df['status'] == 'success').sum() / len(final_full_df) * 100
    print(f"Success rate (GPS extracted): {success_rate:.2f}%")
    
    # Images with GPS coordinates
    gps_data = final_full_df[final_full_df['status'] == 'success'].copy()
    if len(gps_data) > 0:
        print(f"\nCoordinate ranges:")
        print(f"  Latitude: [{gps_data['lat'].min():.4f}, {gps_data['lat'].max():.4f}]")
        print(f"  Longitude: [{gps_data['lon'].min():.4f}, {gps_data['lon'].max():.4f}]")
    
    # Status distribution
    print(f"\nStatus distribution:")
    for status in final_full_df['status'].unique():
        count = (final_full_df['status'] == status).sum()
        pct = count / len(final_full_df) * 100
        print(f"  {status}: {count} ({pct:.2f}%)")
    
    # Dataset distribution
    print(f"\nDataset distribution:")
    for dataset in final_full_df['dataset'].unique():
        count = (final_full_df['dataset'] == dataset).sum()
        pct = count / len(final_full_df) * 100
        print(f"  {dataset}: {count} ({pct:.2f}%)")

if final_italy_df is not None and len(final_italy_df) > 0:
    print(f"\n--- Italy Dataset Analysis ---")
    print(f"Total Italy landmarks: {len(final_italy_df)}")
    
    italy_gps = final_italy_df[final_italy_df['status'] == 'success'].copy()
    if len(italy_gps) > 0:
        print(f"\nItaly coordinate ranges:")
        print(f"  Latitude: [{italy_gps['lat'].min():.4f}, {italy_gps['lat'].max():.4f}]")
        print(f"  Longitude: [{italy_gps['lon'].min():.4f}, {italy_gps['lon'].max():.4f}]")
        
        print(f"\nItaly landmarks by dataset:")
        for dataset in italy_gps['dataset'].unique():
            count = (italy_gps['dataset'] == dataset).sum()
            print(f"  {dataset}: {count}")
        
        print(f"\nItaly landmarks by shard (top 10 shards):")
        shard_counts = italy_gps['shard'].value_counts().head(10)
        for shard, count in shard_counts.items():
            print(f"  Shard {shard:03d}: {count} landmarks")

print(f"\n--- Recommended Next Steps ---")
print(f"1. Verify the data in Google Drive: {project_folder}")
print(f"2. Download images using the file paths in the CSV files")
print(f"3. Use the 'is_italy' column to filter for Italy-specific landmarks")
print(f"4. Use the 'dataset' column to track which dataset each image comes from")
print(f"5. Use the 'shard' column to track which shard/section each image belongs to")


## Shard and Extracted Cache System

### How the Cache Works

The notebook uses an intelligent caching system to avoid re-downloading shards and re-extracting images:

#### 1. **Shard Location**
All downloaded shards are stored in:
```
/content/drive/My Drive/Vision_Project_2026/GLDv2/archives/
```

Shards are named as: `{dataset}_{shard_number}.tar`
- Example: `train_images_000.tar`, `index_images_042.tar`, `test_images_005.tar`

#### 2. **Cache Check Logic**
Before downloading, the notebook checks:
```python
if os.path.exists(local_tar):
    return local_tar  # Use cached version
```

#### 3. **Download Status Tracking**
During processing, the notebook tracks:
- **✓ CACHED**: Shard already downloaded (reused)
- **⬇️ DOWNLOADED**: Newly downloaded this run
- **Total Size**: Cumulative size of all processed shards

#### 4. **Cache Statistics**
At the end of each dataset section, you'll see:
```
[TRAIN DATASET SUMMARY]
  Downloaded: 5 new shards (4.82 GB)
  Cached: 45 shards (reused)
  Total processed: 45,000 images
```

### Disk Space Requirements

- **Train dataset**: ~500 GB (500 shards × ~1 GB each)
- **Index dataset**: ~85 GB (100 shards × ~850 MB each)
- **Test dataset**: ~10 GB (20 shards × ~500 MB each)
- **Extracted images** (persistent cache): ~100 GB+ (kept by default for faster resume)
- **CSV output files**: ~1-2 GB

**Total**: ~600-700 GB if keeping all archives

### Speed Optimization

If you're re-running the notebook:
- Cached shards load instantly (no re-download)
- Only new shards are downloaded
- Resume from exactly where you left off
- No duplicate processing

### Manual Cache Management

To view cached shards:
```bash
ls -lh /content/drive/My\ Drive/Vision_Project_2026/GLDv2/archives/
du -sh /content/drive/My\ Drive/Vision_Project_2026/GLDv2/archives/
```

To delete cache and start fresh:
```bash
rm -rf /content/drive/My\ Drive/Vision_Project_2026/GLDv2/archives/*
```

To keep only specific datasets (e.g., train only):
```bash
rm /content/drive/My\ Drive/Vision_Project_2026/GLDv2/archives/index_*
rm /content/drive/My\ Drive/Vision_Project_2026/GLDv2/archives/test_*
```

In [ ]:
# @title 8. RESUMABILITY AND TROUBLESHOOTING GUIDE

print("=" * 80)
print("RESUMABILITY AND TROUBLESHOOTING GUIDE")
print("=" * 80)

print("""
## How Resumability Works

This notebook is designed to be fully resumable in case of interruption:

1. **Processed ID Tracking**: All processed image IDs are stored in the CSV files.
   When you restart, the notebook automatically loads these IDs and skips them.

2. **Incremental CSV Writing**: Records are saved to CSV every SAVE_INTERVAL images,
   so you don't lose work if interrupted mid-shard.

3. **Shard Caching**: Downloaded TAR files are cached in the archive directory.
   Re-running won't re-download already-downloaded shards.

4. **CSV Append Mode**: New records are appended to existing CSV files,
   preserving all previously collected data.

## To Resume Processing

Simply run this notebook again from the beginning. It will:
- Skip all previously processed images (tracked by image_id in CSV)
- Continue from where it left off
- Append new records to the existing CSV files

## Troubleshooting Tips

1. **If GPS extraction fails for many images**:
   - Wikimedia might be rate limiting. Increase BATCH_PAUSE_SECONDS
   - Some landmarks might not have GeoHack links. This is expected.
   - Check the 'status' column - 'no_gps_found' is normal for some images.

2. **If disk space is running out**:
   - Set DELETE_EXTRACTED_AFTER_SHARD = True to free space after each shard
   - Set KEEP_TAR_ARCHIVES = False to delete downloaded TAR files

3. **If you want to restart from scratch**:
   - Delete the CSV files in the project folder
   - Optionally delete the archive directory to re-download all shards

4. **To process only specific shards for testing**:
   - Set START_SHARD = 0 and MAX_SHARDS = 5 (for example)
   - Re-run the notebook to test with just 5 shards

5. **To process only specific datasets**:
   - Set PROCESS_TRAIN = False, PROCESS_INDEX = False, or PROCESS_TEST = False
   - Only enabled datasets will be processed

## CSV Output Format

Both output CSV files (full dataset and Italy-only) contain:

- **image_id**: 16-character image identifier
- **file_path**: Full path to the extracted image file
- **wiki_url**: Wikimedia URL used to find GPS coordinates
- **lat**: Latitude coordinate (None if not found)
- **lon**: Longitude coordinate (None if not found)
- **status**: Processing status ('success', 'no_gps_found', 'missing_metadata', etc.)
- **dataset**: Which dataset this image belongs to ('train', 'index', or 'test')
- **shard**: Which shard/section this image belongs to (0-499 for train, 0-99 for index, etc.)
- **is_italy**: Whether coordinates fall within Italy's bounding box (True/False)

## Performance Tips

1. Increase MAX_WORKERS to 16-20 for faster GPS extraction (if network allows)
2. Increase URL_BATCH_SIZE to 5000 for fewer batches (but larger latency between batches)
3. Increase SAVE_INTERVAL to 500 for less frequent CSV writes (faster processing)
4. Set PROCESS_TRAIN = True, PROCESS_INDEX = False, PROCESS_TEST = False if you only need one dataset
""")

print("\\n" + "=" * 80)
print("Setup complete! The notebook is ready for production use.")
print("=" * 80)


## Performance Optimization Guide

### How Fast is the Scraping?

#### Speed Factors

The GPS scraping speed depends on:
1. **Network speed** - Download bandwidth (primary bottleneck)
2. **Wikimedia responsiveness** - GeoHack server latency (0.2-1.5 sec per request)
3. **Thread count** - Parallel requests (MAX_WORKERS setting)
4. **Batch size** - URLs processed per pause interval
5. **Your location** - Geographic proximity to servers

#### Estimated Time for Full Dataset

| Dataset | Images | Unique URLs | Estimated Time |
|---------|--------|-------------|-----------------|
| Train | 4.1M | ~1.2M | **15-25 hours** |
| Index | 761K | ~250K | **3-5 hours** |
| Test | 117K | ~5K | **0.5-1 hour** |
| **TOTAL** | **~5M** | **~1.5M** | **18-31 hours** |

**Note**: Times assume continuous processing. With DEFAULT settings (MAX_WORKERS=12), expect ~24-31 hours. With OPTIMIZED settings (MAX_WORKERS=20), expect ~18-24 hours.

### Optimization Strategies

#### 1. **Increase Thread Workers** (FASTEST IMPACT)
```python
MAX_WORKERS = 20  # Increase from 12 to 20
```
- **Speed**: +40% faster (roughly)
- **Risk**: Low if your connection is stable
- **Trade-off**: Uses more memory and bandwidth

#### 2. **Larger URL Batches** (FAST)
```python
URL_BATCH_SIZE = 5000  # Increased from 2000
```
- **Speed**: +20% faster
- **Why**: Fewer pause intervals between batches
- **Trade-off**: Uses more memory during batch processing
- **Formula**: More URLs = fewer pause intervals = faster throughput

#### 3. **Reduce Batch Pause** (MODERATE SPEED)
```python
BATCH_PAUSE_SECONDS = 0.05  # Reduced from 0.2
```
- **Speed**: +30% faster
- **Risk**: May trigger Wikimedia rate limiting (429 errors)
- **If you get errors**: Increase back to 0.1-0.2

#### 4. **Connection Pooling** (ALREADY ENABLED) ✓
Already optimized in this version with:
- `requests.Session()` for TCP connection reuse
- `HTTPAdapter` with connection pooling
- **Speed**: +15-20% faster for repeated requests
- **Benefit**: Automatic, no configuration needed

#### 5. **Reduce Request Timeouts** (RISKY)
```python
REQUEST_TIMEOUT_SECONDS = 8  # Reduced from 10
REQUEST_RETRIES = 1  # Reduced from 2
```
- **Speed**: +10-15% faster
- **Risk**: May fail for slow connections or timeouts
- **Recommendation**: Only if your connection is fast

### Optimization Profiles

Choose based on your needs:

**CONSERVATIVE (Stable)**
```python
MAX_WORKERS = 12
URL_BATCH_SIZE = 2000
BATCH_PAUSE_SECONDS = 0.2
REQUEST_TIMEOUT_SECONDS = 10
REQUEST_RETRIES = 2
# Estimated time: 25-35 hours
# Reliability: Very high
```

**BALANCED (Recommended)**
```python
MAX_WORKERS = 16
URL_BATCH_SIZE = 5000
BATCH_PAUSE_SECONDS = 0.1
REQUEST_TIMEOUT_SECONDS = 10
REQUEST_RETRIES = 1
# Estimated time: 18-24 hours
# Reliability: High
```

**AGGRESSIVE (Fast)**
```python
MAX_WORKERS = 20
URL_BATCH_SIZE = 8000
BATCH_PAUSE_SECONDS = 0.05
REQUEST_TIMEOUT_SECONDS = 8
REQUEST_RETRIES = 1
# Estimated time: 14-20 hours
# Reliability: Medium (expect some rate limiting)
```

**VERY AGGRESSIVE (Fastest)**
```python
MAX_WORKERS = 24
URL_BATCH_SIZE = 10000
BATCH_PAUSE_SECONDS = 0.02
REQUEST_TIMEOUT_SECONDS = 6
REQUEST_RETRIES = 0
# Estimated time: 12-18 hours
# Reliability: Lower (expect failures)
```

### Real-World Performance

#### With Default Settings (MAX_WORKERS=12)
- GPS fetch rate: ~500-800 URLs/minute
- For 1.5M URLs: ~30-50 hours
- Reason: Wikimedia GeoHack is slow (~0.5-1.5 sec per request)

#### With Optimized Settings (MAX_WORKERS=16-20)
- GPS fetch rate: ~800-1200 URLs/minute
- For 1.5M URLs: ~18-25 hours
- Connection pooling saves ~15% overhead

### Troubleshooting Performance

If you see **many 429 errors** (rate limiting):
```python
BATCH_PAUSE_SECONDS = 0.2  # Increase pause
URL_BATCH_SIZE = 2000  # Reduce batch size
MAX_WORKERS = 12  # Reduce workers
```

If you see **timeout errors**:
```python
REQUEST_TIMEOUT_SECONDS = 12  # Increase timeout
BATCH_PAUSE_SECONDS = 0.15  # Give more time between batches
```

If you have **low bandwidth**:
```python
MAX_WORKERS = 8  # Use fewer threads
BATCH_PAUSE_SECONDS = 0.5  # Longer pauses
```

If you want **maximum speed**:
```python
# Already using connection pooling
# Increase MAX_WORKERS to 20-24
# Reduce BATCH_PAUSE_SECONDS to 0.05
# Accept some failures (they'll show as 'no_gps_found' status)
```

### Cost-Benefit Analysis

| Change | Speed Gain | Effort | Risk |
|--------|-----------|--------|------|
| MAX_WORKERS: 12→20 | **+40%** | Easy | Low |
| URL_BATCH_SIZE: 2K→5K | **+20%** | Easy | Very Low |
| BATCH_PAUSE: 0.2→0.1 | **+30%** | Easy | Medium |
| Connection pooling | **+15%** | Done ✓ | None |
| REQUEST_TIMEOUT: 10→8 | **+10%** | Easy | High |
| **TOTAL POSSIBLE** | **~+115%** | Easy | Medium |

**Bottom line**: Adjusting `MAX_WORKERS` and `BATCH_PAUSE_SECONDS` gives best speed/reliability ratio.

In [ ]:
# @title 9. RUNTIME MONITORING AND PROGRESS ESTIMATION

import time as time_module

def estimate_remaining_time(start_time, processed_count, total_count):
    """Estimate remaining processing time."""
    elapsed = time_module.time() - start_time
    if processed_count == 0:
        return None
    
    rate = processed_count / elapsed  # items per second
    remaining = total_count - processed_count
    eta_seconds = remaining / rate if rate > 0 else None
    
    return eta_seconds

def format_time(seconds):
    """Format seconds to readable time format."""
    if seconds is None:
        return "N/A"
    
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    
    if hours > 0:
        return f"{hours}h {minutes}m {secs}s"
    elif minutes > 0:
        return f"{minutes}m {secs}s"
    else:
        return f"{secs}s"

def calculate_processing_rate(dataset_name, images_count, start_time, current_time=None):
    """Calculate and display processing rate."""
    if current_time is None:
        current_time = time_module.time()
    
    elapsed = current_time - start_time
    if elapsed == 0:
        return 0
    
    rate = images_count / elapsed
    print(f"\n[{dataset_name}] Processing Rate: {rate:.1f} images/sec ({rate*60:.0f} images/min)")
    return rate

# Example usage (can be called during processing):
print("""
RUNTIME MONITORING EXAMPLE:

During processing, you can monitor progress like this:

    # At the start of processing
    start_time = time.time()
    
    # After processing N images
    eta = estimate_remaining_time(start_time, 50000, 4132914)
    print(f"Estimated time remaining: {format_time(eta)}")
    
    # Check processing rate
    rate = calculate_processing_rate("Train", 50000, start_time)
    
To get a live progress estimate during the actual run, look at:
- The tqdm progress bars showing iteration count
- The '[Train Dataset Summary]' output at the end of each dataset
- The total rows in the CSV files

QUICK MATH:
- If processing 100K images takes 2 hours = 13.9 images/sec
- Remaining 4M images / 13.9 images/sec = 81 hours more
- Total: ~83 hours (or ~3.5 days)

But with optimization (MAX_WORKERS=20):
- Same 100K images takes ~1.2 hours = 23 images/sec
- Remaining 4M / 23 = 48 hours more  
- Total: ~49 hours (or ~2 days)

You can check actual progress in the CSV files:
    df = pd.read_csv(gld_full_dataset.csv)
    processed = len(df)
    print(f"Progress: {processed:,} / 4,132,914 images ({processed/4132914*100:.1f}%)")
""")

print("✓ Monitoring functions ready!")